In [ ]:
%pip install powerlaw networkx numpy scipy pandas tqdm scipy

In [ ]:
import sys
import logging
from pathlib import Path

# sys.path manipulation is required because sdt_netval is not installed as a package
src_path = str(Path("../src").resolve())
if src_path not in sys.path:
    sys.path.insert(0, src_path)

# scripts/ must be on sys.path for dynamic imports of visualizer classes
scripts_path = str(Path("../scripts").resolve())
if scripts_path not in sys.path:
    sys.path.insert(0, scripts_path)

logging.basicConfig(level=logging.INFO, format="%(name)s — %(message)s")

from sdt_netval import load_network

# Carica un singolo run di test (legacy Tomašević)
db_path = Path("../data/00_raw/01_legacy_tomasevic/benchmark_runs/run01.sqlite").resolve()
G = load_network(db_path)

In [ ]:
import pandas as pd
from sdt_netval import GraphMetrics

report = GraphMetrics(G).generate_full_report()

pd.Series(report).rename("value").to_frame()

In [ ]:
from sdt_netval.pipeline import StageAValidator

data_dir = Path("../data/00_raw/01_legacy_tomasevic/benchmark_runs").resolve()

validator = StageAValidator(data_dir)
validator.process_runs()
validator.save_raw_results(Path("../data/01_processed/01_legacy_tomasevic/stage_a_raw.csv").resolve())

report = validator.full_stability_report()
print("\n=== Topological Stability (Stage A — Legacy Tomašević) ===\n")
print(report["stability"])

In [ ]:
from importlib import import_module

stage_a_viz = import_module("02_stage_a_visualization")
StageAVisualizer = stage_a_viz.StageAVisualizer

viz = StageAVisualizer("../data/01_processed/01_legacy_tomasevic/stage_a_raw.csv")
viz.plot_stability_distributions("../data/01_processed/01_legacy_tomasevic/stage_a_stability.png")
print(viz.plot_summary_statistics())

In [ ]:
from sdt_netval.pipeline import StageBAnalyzer

analyzer = StageBAnalyzer(Path("../data/00_raw/01_legacy_tomasevic/sensitivity_runs").resolve())
analyzer.process_all_runs()
analyzer.save_raw_results(Path("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv").resolve())
analyzer.save_aggregated_results(Path("../data/01_processed/01_legacy_tomasevic/stage_b_aggregated.csv").resolve())

report = analyzer.full_sensitivity_report()
print(report["aggregated"])

In [ ]:
from importlib import import_module

stage_b_viz = import_module("04_stage_b_visualization")
StageBVisualizer = stage_b_viz.StageBVisualizer

viz = StageBVisualizer("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv")
viz.plot_modularity_comparison("../data/01_processed/01_legacy_tomasevic/stage_b_modularity_comparison.png")
print(viz.get_summary_statistics())

In [ ]:
from importlib import import_module

stage_b_hyp = import_module("05_stage_b_hypothesis")
StageBHypothesisTesting = stage_b_hyp.StageBHypothesisTesting

tester = StageBHypothesisTesting("../data/01_processed/01_legacy_tomasevic/stage_b_raw.csv")
results_df = tester.run_tests()

tester.print_results(verbose=True)
tester.save_results("../data/01_processed/01_legacy_tomasevic/stage_b_pvalues.csv")

sig_conditions = tester.get_significant_conditions(alpha=0.05)
print(f"Significant conditions (p<0.05): {sig_conditions}")